In [ ]:
import os
import openai
import yaml
import zipfile
from functools import lru_cache
import uuid
import json

In [2]:
knwf_path = "data/file_to_translate/2024_nuclei_segmentation_knime.knwf"
extract_dir = "data/file_to_translate_unzipped"

1. Extract a KNIME .knwf file into the target directory.

In [ ]:

def unzip_knime_workflow(zip_path: str, extract_to: str):
    """
    Unpacks a KNIME .knwf file into the specified target directory.
    """
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f"Unzip: {zip_path} → {extract_to}")

unzip_knime_workflow(knwf_path, extract_dir)

Entpackt: data/file_to_translate/2024_nuclei_segmentation_knime.knwf → data/file_to_translate_unzipped


2. Workflow-Verzeichnis finden

In [ ]:
def find_workflow_dir(base_dir: str) -> str:
    """
    Searches the base_dir directory for the first subfolder that contains a file named 'workflow.knime'. It then returns the full path to this subfolder.
    """
    for entry in os.listdir(base_dir):  # Iterate through all files and folders in base_dir.
        full_path = os.path.join(base_dir, entry)

        # If it is a folder and contains 'workflow.knime' → success.
        if os.path.isdir(full_path) and "workflow.knime" in os.listdir(full_path):
            print(f"Workflow directory found: {full_path}")
            return full_path

    # If no such folder is found → raise an error.
    raise ValueError(f"Non-valid workflow directory {base_dir}")

workflow_dir = find_workflow_dir(extract_dir)


Workflow-Ordner gefunden: data/file_to_translate_unzipped/2024_nuclei_segmentation_knime


3. Sammelt alle settings.xml-Dateien aus den Node-Unterordnern und gibt sie als dict zurück.
    Key: Node-Name (z. B. "Image Reader (#1)")
    Value: XML-String

In [ ]:
def collect_knime_node_files(workflow_dir: str):
    """
    Collects all settings.xml files from the node subfolders and returns them as a dict.
    Key: Node name (e.g., "Image Reader (#1)")
    Value: XML string
    """
    node_data = {}

    for entry in os.listdir(workflow_dir):
        full_path = os.path.join(workflow_dir, entry)
        if os.path.isdir(full_path) and not entry.startswith('.'):
            xml_path = os.path.join(full_path, "settings.xml")
            if os.path.exists(xml_path):
                with open(xml_path, 'r', encoding='utf-8') as f:
                    xml_content = f.read()
                    node_data[entry] = xml_content

    return node_data

In [6]:
knime_nodes = collect_knime_node_files(workflow_dir)
knime_nodes

{'Splitter (#2)': '<?xml version="1.0" encoding="UTF-8"?>\n<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="settings.xml">\n    <entry key="node_file" type="xstring" value="settings.xml"/>\n    <config key="flow_stack"/>\n    <config key="internal_node_subsettings">\n        <entry key="memory_policy" type="xstring" value="CacheSmallInMemory"/>\n    </config>\n    <config key="model">\n        <config key="column_selection_Internals">\n            <entry key="SettingsModelID" type="xstring" value="SMID_string"/>\n            <entry key="EnabledStatus" type="xboolean" value="true"/>\n        </config>\n        <entry key="column_selection" type="xstring" value=""/>\n        <config key="dim_selection_Internals">\n            <entry key="SettingsModelID" type="xstring" value="SMID_dimselection"/>\n            <entry key="E

In [ ]:
# Collects the workflow.knime file
def collect_workflow_file(workflow_dir: str) -> str:
    """
    Collects the content of the workflow.knime file and returns it as a string.
    """
    workflow_path = os.path.join(workflow_dir, "workflow.knime")
    if not os.path.exists(workflow_path):
        raise FileNotFoundError(f"File {workflow_path} not found.")
    
    with open(workflow_path, 'r', encoding='utf-8') as f:
        return f.read()


In [ ]:
# Output the first 500 characters of the workflow
workflow_content = collect_workflow_file(workflow_dir)
print(workflow_content[:500])  

<?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="workflow.knime">
    <entry key="created_by" type="xstring" value="5.1.2.v202310091347"/>
    <entry key="created_by_nightly" type="xboolean" value="false"/>
    <entry key="version" type="xstring" value="5.1.0"/>
    <entry key="name" type="xs


In [ ]:
knime_nodes = collect_knime_node_files(workflow_dir)
print("First 100 characters in every node:")
for node_name, xml_content in knime_nodes.items():
    print(f"{node_name}: {xml_content[:100]}...")

Erste 100 Zeichen jeder Node:
Splitter (#2): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
Gaussian Convolution (#3): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
Image Reader (#1): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
Image Features (#7): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
Global Thresholder (#5): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
Connected Component Analysis (#6): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
CLAHE (#4): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...


In [10]:
@lru_cache(maxsize=1)
def load_translation_examples(yaml_path: str = "data/translation_table.yml") -> list:
    with open(yaml_path, "r", encoding="utf-8") as f:
        docs = list(yaml.safe_load_all(f))
        print(docs)
        examples = []
        for doc in docs[0]:
            knime = doc.get("KNIME")
            galaxy = doc.get("Galaxy")

            if knime and galaxy:
                examples.append({
                    "KNIME": knime.strip(),
                    "Galaxy": galaxy.strip()
                })

    return examples


In [11]:
translation_examples = load_translation_examples(yaml_path="data/translation_table.yml")
translation_examples

[[{'description': 'Input Image dataset', 'Galaxy': '"0": {\n        "annotation": "",\n        "content_id": null,\n        "errors": null,\n        "id": 0,\n        "input_connections": {},\n        "inputs": [\n            {\n                "description": "",\n                "name": "Input Image Dataset"\n            }\n        ],\n        "label": "Input Image Dataset",\n        "name": "Input dataset",\n        "outputs": [],\n        "position": {\n            "left": 0.0,\n            "top": 0.0\n        },\n        "tool_id": null,\n        "tool_state": "{\\"optional\\": false, \\"tag\\": null}",\n        "tool_version": null,\n        "type": "data_input",\n        "uuid": "312178b3-3fac-49ab-95cf-29e3f44e6015",\n        "when": null,\n        "workflow_outputs": []\n    },\n', 'Python': "from skimage import io\n\ninput_image = io.imread('input_image.tiff')\n", 'KNIME': '<?xml version="1.0" encoding="UTF-8"?>\n<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi

[{'KNIME': '<?xml version="1.0" encoding="UTF-8"?>\n<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="settings.xml">\n    <entry key="node_file" type="xstring" value="settings.xml"/>\n    <config key="flow_stack"/>\n    <config key="internal_node_subsettings">\n        <entry key="memory_policy" type="xstring" value="CacheSmallInMemory"/>\n    </config>\n    <config key="model">\n        <config key="check_file_format_Internals">\n            <entry key="SettingsModelID" type="xstring" value="SMID_boolean"/>\n            <entry key="EnabledStatus" type="xboolean" value="true"/>\n        </config>\n        <entry key="check_file_format" type="xboolean" value="true"/>\n        <config key="group_files_Internals">\n            <entry key="SettingsModelID" type="xstring" value="SMID_boolean"/>\n            <entry key="Enabled

In [12]:
def build_examples():
    examples_text =  """You are a translator that converts KNIME workflow nodes to Galaxy workflow steps.

Below are examples of how this translation should be done:
"""
    examples = load_translation_examples()
    if examples:
        for i, ex in enumerate(examples[:6]):  # z. B. nur 6 Beispiele
            examples_text += f"""

## Example {i + 1}:

KNIME node (XML):

```xml
{ex["KNIME"]}
```

Galaxy step (JSON):
```json
{ex["Galaxy"]}
```

"""
    return examples_text


In [13]:
node_examples = build_examples()
print(node_examples)

[[{'description': 'Input Image dataset', 'Galaxy': '"0": {\n        "annotation": "",\n        "content_id": null,\n        "errors": null,\n        "id": 0,\n        "input_connections": {},\n        "inputs": [\n            {\n                "description": "",\n                "name": "Input Image Dataset"\n            }\n        ],\n        "label": "Input Image Dataset",\n        "name": "Input dataset",\n        "outputs": [],\n        "position": {\n            "left": 0.0,\n            "top": 0.0\n        },\n        "tool_id": null,\n        "tool_state": "{\\"optional\\": false, \\"tag\\": null}",\n        "tool_version": null,\n        "type": "data_input",\n        "uuid": "312178b3-3fac-49ab-95cf-29e3f44e6015",\n        "when": null,\n        "workflow_outputs": []\n    },\n', 'Python': "from skimage import io\n\ninput_image = io.imread('input_image.tiff')\n", 'KNIME': '<?xml version="1.0" encoding="UTF-8"?>\n<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi

In [ ]:
# Convert dict content to string.
knime_nodes_str = "\n".join(
    f"Node ID: {key}\n{value}" for key, value in knime_nodes.items()
)
print(knime_nodes_str)

Node ID: Splitter (#2)
<?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="settings.xml">
    <entry key="node_file" type="xstring" value="settings.xml"/>
    <config key="flow_stack"/>
    <config key="internal_node_subsettings">
        <entry key="memory_policy" type="xstring" value="CacheSmallInMemory"/>
    </config>
    <config key="model">
        <config key="column_selection_Internals">
            <entry key="SettingsModelID" type="xstring" value="SMID_string"/>
            <entry key="EnabledStatus" type="xboolean" value="true"/>
        </config>
        <entry key="column_selection" type="xstring" value=""/>
        <config key="dim_selection_Internals">
            <entry key="SettingsModelID" type="xstring" value="SMID_dimselection"/>
            <entry key="EnabledStatu

In [15]:
def build_task_prompt():
    
    task_prompt = f"""
# Your Task
You are a system that translates complete KNIME workflows into Galaxy workflows. Produce a **single, valid Galaxy .ga workflow JSON** that can be imported in Galaxy, representing the entire KNIME workflow below.

# Input
KNIME Nodes (XML):
```xml
{knime_nodes_str}
```

The KNIME workflow content (XML):
```xml
{workflow_content}
```

# Output Requirements
- Respond with the complete Galaxy workflow JSON object ONLY (no markdown fences, no comments, no explanations).
- The JSON must be a valid Galaxy .ga workflow 
- Make sure that it is a valid JSON object.
- For uuid fields, write 00000000-0000-0000-0000-000000000000 as placeholder
- Do not include TODOs or comments in the JSON.

"""
    
    return task_prompt

In [16]:
task = build_task_prompt()
print(task)


# Your Task
You are a system that translates complete KNIME workflows into Galaxy workflows. Produce a **single, valid Galaxy .ga workflow JSON** that can be imported in Galaxy, representing the entire KNIME workflow below.

# Input
KNIME Nodes (XML):
```xml
Node ID: Splitter (#2)
<?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="settings.xml">
    <entry key="node_file" type="xstring" value="settings.xml"/>
    <config key="flow_stack"/>
    <config key="internal_node_subsettings">
        <entry key="memory_policy" type="xstring" value="CacheSmallInMemory"/>
    </config>
    <config key="model">
        <config key="column_selection_Internals">
            <entry key="SettingsModelID" type="xstring" value="SMID_string"/>
            <entry key="EnabledStatus" type="xboolean" valu

In [17]:
def build_workflow_examples():
    workflow_examples_text =  """
    Here are some examples of complete KNIME workflows and their corresponding Galaxy workflows:
    """  
    examples = load_translation_examples(yaml_path="data/workflow_translation_table.yml")
    if examples:
        for i, ex in enumerate(examples[:]): 
            workflow_examples_text += f"""

## Example {i + 1}:

KNIME node (XML):

```xml
{ex["KNIME"]}
```

Galaxy step (JSON):
```json
{ex["Galaxy"]}
```

"""
    return workflow_examples_text

In [18]:
workflow_examples = build_workflow_examples()
print(workflow_examples)

[[{'description': 'Example 1 resize rotate', 'KNIME': '<?xml version="1.0" encoding="UTF-8"?>\n<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="workflow.knime">\n    <entry key="created_by" type="xstring" value="4.7.7.v202308161346"/>\n    <entry key="created_by_nightly" type="xboolean" value="false"/>\n    <entry key="version" type="xstring" value="4.1.0"/>\n    <entry key="name" type="xstring" isnull="true" value=""/>\n    <config key="authorInformation">\n        <entry key="authored-by" type="xstring" value="massei"/>\n        <entry key="authored-when" type="xstring" value="2025-08-25 13:02:31 +0200"/>\n        <entry key="lastEdited-by" type="xstring" value="massei"/>\n        <entry key="lastEdited-when" type="xstring" value="2025-08-25 13:10:11 +0200"/>\n    </config>\n    <entry key="customDescription" type="xst

In [ ]:
full_prompt = f"{node_examples}\n\n{workflow_examples}\n\n{task}"
print(full_prompt)

You are a translator that converts KNIME workflow nodes to Galaxy workflow steps.

Below are examples of how this translation should be done:


## Example 1:

KNIME node (XML):

```xml
<?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="settings.xml">
    <entry key="node_file" type="xstring" value="settings.xml"/>
    <config key="flow_stack"/>
    <config key="internal_node_subsettings">
        <entry key="memory_policy" type="xstring" value="CacheSmallInMemory"/>
    </config>
    <config key="model">
        <config key="check_file_format_Internals">
            <entry key="SettingsModelID" type="xstring" value="SMID_boolean"/>
            <entry key="EnabledStatus" type="xboolean" value="true"/>
        </config>
        <entry key="check_file_format" type="xboolean" value="true"

In [20]:
def list_scadsai_models():
    client = openai.OpenAI(
        base_url="https://llm.scads.ai/v1",
        api_key=os.environ.get("SCADSAI_API_KEY")
    )
    models = client.models.list()
    return [m.id for m in models.data]

In [21]:
list_scadsai_models()

['alias-code',
 'alias-reasoning',
 'alias-image-generation',
 'alias-vision',
 'alias-ha',
 'meta-llama/Llama-3.3-70B-Instruct',
 'meta-llama/Llama-3.1-8B-Instruct',
 'Qwen/Qwen2-VL-7B-Instruct',
 'openGPT-X/Teuken-7B-instruct-research-v0.4',
 'deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct',
 'Qwen/Qwen3-Coder-30B-A3B-Instruct',
 'Qwen/Qwen3-Embedding-4B',
 'stabilityai/stable-diffusion-3.5-large-turbo',
 'black-forest-labs/FLUX.1-dev',
 'openai/whisper-large-v3',
 'Kokoro-82M',
 'tts-1-hd',
 'deepseek-ai/DeepSeek-R1',
 'meta-llama/Llama-4-Scout-17B-16E-Instruct',
 'openai/gpt-oss-120b']

In [22]:
def prompt_scadsai_llm(message:str, model="openai/gpt-oss-120b"):
    """A prompt helper function that sends a message to ScaDS.AI LLM server at 
    ZIH TU Dresden and returns only the text response.
    """
    
    # convert message in the right format if necessary
    if isinstance(message, str):
        message = [{"role": "user", "content": message}]
    
    # setup connection to the LLM
    client = openai.OpenAI(base_url="https://llm.scads.ai/v1",
                           api_key=os.environ.get('SCADSAI_API_KEY')
    )
    response = client.chat.completions.create(
        model=model,
        messages=message
    )
    
    # extract answer
    return response.choices[0].message.content

In [23]:
answer = prompt_scadsai_llm(message= full_prompt)
print(answer)

{
  "a_galaxy_workflow": "true",
  "annotation": "",
  "comments": [],
  "creator": [
    {
      "class": "Person",
      "identifier": "",
      "name": "Riccardo"
    }
  ],
  "format-version": "0.1",
  "license": "MIT",
  "name": "knime_to_galaxy_workflow",
  "readme": "Converted from KNIME workflow.",
  "steps": {
    "0": {
      "annotation": "",
      "content_id": null,
      "errors": null,
      "id": 0,
      "input_connections": {},
      "inputs": [],
      "label": null,
      "name": "Input dataset",
      "outputs": [],
      "position": {
        "left": 0.0,
        "top": 0.0
      },
      "post_job_actions": {},
      "tool_id": null,
      "tool_state": "{\"optional\": false, \"tag\": null}",
      "tool_version": null,
      "type": "data_input",
      "uuid": "00000000-0000-0000-0000-000000000000",
      "when": null,
      "workflow_outputs": []
    },
    "1": {
      "annotation": "",
      "content_id": "toolshed.g2.bx.psu.edu/repos/imgteam/bfconvert/ip_con

In [24]:
# Parse the answer to a JSON object

try: 
    json_object = json.loads(answer)
    print("Parsed JSON successfully.")
    print(json_object)
except json.JSONDecodeError as e:
    print("Failed to parse JSON:", e)


Parsed JSON successfully.
{'a_galaxy_workflow': 'true', 'annotation': '', 'comments': [], 'creator': [{'class': 'Person', 'identifier': '', 'name': 'Riccardo'}], 'format-version': '0.1', 'license': 'MIT', 'name': 'knime_to_galaxy_workflow', 'readme': 'Converted from KNIME workflow.', 'steps': {'0': {'annotation': '', 'content_id': None, 'errors': None, 'id': 0, 'input_connections': {}, 'inputs': [], 'label': None, 'name': 'Input dataset', 'outputs': [], 'position': {'left': 0.0, 'top': 0.0}, 'post_job_actions': {}, 'tool_id': None, 'tool_state': '{"optional": false, "tag": null}', 'tool_version': None, 'type': 'data_input', 'uuid': '00000000-0000-0000-0000-000000000000', 'when': None, 'workflow_outputs': []}, '1': {'annotation': '', 'content_id': 'toolshed.g2.bx.psu.edu/repos/imgteam/bfconvert/ip_convertimage/6.7.0+galaxy3', 'errors': None, 'id': 1, 'input_connections': {'input_file': {'id': 0, 'output_name': 'output'}}, 'inputs': [], 'label': None, 'name': 'Extract Channel', 'outputs'

In [33]:
if "uuid" in json_object:
    json_object["uuid"] = str(uuid.uuid4())

In [29]:
# Go through the JSON object recursively searching for the "uuid" key and replace the correspondign value with a new UUID
# Use the uuid module to generate a new UUID
for step in json_object["steps"].values():
    if isinstance(step, dict) and "uuid" in step:
        step["uuid"] = str(uuid.uuid4())

In [34]:
print(json_object["uuid"])
print([s["uuid"] for s in json_object["steps"].values()])

b40f82af-00b9-42db-a2ee-2547c2fc6937
['4a0d3cdf-5eb3-40e3-92cc-283f6273a956', '20d6594d-85fd-4a77-a2da-651024f787da', 'e301ef5e-6851-443d-8035-d67a9bcfdccf', 'f36d5163-9b16-476b-9969-83d7ea55eac4', 'be93e6c9-3f02-4c52-a6fd-87156c7270ab', '00940301-2aed-4ac9-b9cc-70a6e994bf23', '3a480827-b75b-46e5-bca1-1bb805f6b0be']


In [37]:
# Save the answer to a file
output_file = "data/output_file/knime2galaxy_workflow_output.ga"
with open(output_file, "w", encoding="utf-8") as f:
      json.dump(json_object, f, indent=2, ensure_ascii=False)

print(f"Galaxy workflow saved to {output_file}")

Galaxy workflow saved to data/output_file/knime2galaxy_workflow_output.ga
